# Appendix 1 — Drawing the Benchmark Sample

Model-coded results are only as good as the reference standard they are validated against, so the
study begins with a benchmark: 600 decisions drawn at random from the corpus and categorized one
case at a time by a state-of-the-art frontier model (Anthropic's Claude Fable 5, with Claude Code as
a harness — the process is described at the end of this notebook). The benchmark has one narrow
purpose: to test whether the smaller, cheaper model that categorizes the full corpus (Appendix 2)
produces the same answers as the strongest available model. The 600 cases were drawn in two
batches of 300 for workflow reasons but are analyzed throughout as one sample. This notebook
reproduces the random draws exactly and verifies them against the committed reference file,
`data/golden_ai_generated_sample.json`.

> **A note on file names.** During the study the benchmark was called the “Golden AI Generated
> Sample”, and the committed data files (`golden_ai_generated_sample.json`,
> `golden_predictions_qwen.json`, `golden_sample_coding_rubric.md`, `example_golden_label.json`)
> and the `gold_*` column names keep that working name so that nothing in the research archive
> has to be renamed. Wherever the paper says “benchmark”, these files are what it means.

**Data.** All decisions come from the A2AJ [`canadian-case-law`](https://huggingface.co/datasets/a2aj/canadian-case-law)
dataset on Hugging Face, pinned to one frozen revision so the corpus cannot change under the
study. Four sources: RPD (Refugee Protection Division first-instance decisions), RAD
(Refugee Appeal Division decisions), RLLR (published RPD decisions from the Refugee Law Lab
Reporter), and FC (Federal Court judicial reviews). The English decision text
(`unofficial_text_en`) is the sole coding input.

**Requirements.** Python with the `datasets` and `pandas` packages (see `README.md`), plus
internet access on first run to download the corpus.

AI Disclosure: Generative AI tools were used in preparing these appendices, including for background research, data extraction, coding, drafting and editing (Claude Code, using Fable 5 and 5.5 and Opus 5 and 5.5).


In [1]:
from pathlib import Path
import json
import re
import warnings

import pandas as pd

# cosmetic: silence the progress-bar and unauthenticated-request notices, so the committed
# output carries no local file paths (they change nothing about what is downloaded)
warnings.filterwarnings("ignore", message="IProgress not found")
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()
from datasets import load_dataset

REV = "7ca50f4e7feab57b48223e9252f8aef4d041a29c"   # frozen dataset revision - never change this
SEED = 12                                          # used for ALL random sampling in the study

DATA = Path("data")   # every file these appendices need ships in this data/ folder

## The two draws

The 600 cases were drawn in two deterministic batches. The split mattered only during prompt
development (the second draw stayed out of the tuning loop as an over-fitting check); for all
substantive analyses the 600 cases are one group.

| | Draw 1 (cases `000`–`299`) | Draw 2 (cases `300`–`599`) |
|---|---|---|
| Drawn from | full corpus | decisions dated 2006–2025 (the study window), excluding every draw-1 case |
| Composition | 67 RAD + 67 RPD + 66 RLLR + 100 FC | same |

Draw 1 sampled each source in full — any year — and, for the tribunal sources, did not require
non-empty English text (the FC pool did, through the pre-filter); only draw 2 was limited to
decisions dated 2006–2025 with non-empty text. The cell after the verification below documents
the consequences after the fact: every one of the 600 cases has decision text, and 15 draw-1
cases (10 FC, 5 RPD) are dated outside 2006–2025 and so have no counterpart in the production
corpus of Appendix 3 (which is why Appendices 3 and 4 work with 585 benchmark cases rather than 600).

**Federal Court pre-filter.** FC judicial reviews span every subject (tax, patents,
citizenship…), so before sampling the text must mention a refugee tribunal: `RPD` or `RAD`
(whole words, case-sensitive), `Refugee Protection Division`, or `Refugee Appeal Division`.
About 30% of FC cases pass. A filtered case may still not be a refugee judicial review — that is
what the `jr_refugee` gating question is for. This is the original, narrower pre-filter: both FC
batches of the benchmark were drawn under it. The production run later widened the filter to
also accept `Immigration and Refugee Board` (Appendix 3), so the benchmark contains no decision
admitted only by the widened filter.

The cell below draws both batches; with the dataset revision and seed fixed, re-running always
yields the identical 600 cases.

In [2]:
SIZES  = {"RAD": 67, "RPD": 67, "RLLR": 66, "FC": 100}     # per-source sizes (both samples)
START1 = {"RAD": 0,   "RPD": 67,  "RLLR": 134, "FC": 200}  # sample-1 case numbering: 000-299
START2 = {"RAD": 300, "RPD": 367, "RLLR": 434, "FC": 500}  # sample-2 case numbering: 300-599

FC_FILTER = re.compile(r"\bRPD\b|\bRAD\b|Refugee Protection Division|Refugee Appeal Division")

rows1, rows2, texts = [], [], {}

def record(rows, case_id, d, idx, row, text):
    texts[case_id] = text
    rows.append({"case_id": case_id, "dataset": d, "orig_idx": idx,
                 "citation": row.get("citation_en"), "name": row.get("name_en"),
                 "document_date": row.get("document_date_en"),
                 "source_url": row.get("url_en"), "text_chars": len(text)})

for d in ["RAD", "RPD", "RLLR", "FC"]:
    df = load_dataset("a2aj/canadian-case-law", data_dir=d, split="train", revision=REV).to_pandas()
    df["text"] = df["unofficial_text_en"].fillna("")
    print(f"{d}: {len(df):,} decisions in the corpus")

    # ---- sample 1: whole corpus (FC: text pre-filter) ----
    pool1 = df[df.text.apply(lambda t: bool(FC_FILTER.search(t))) & (df.text.str.len() > 0)] if d == "FC" else df
    s1 = pool1.sample(n=SIZES[d], random_state=SEED)
    for j, (idx, row) in enumerate(s1.iterrows()):
        record(rows1, f"{START1[d] + j:03d}_{d}", d, idx, row, str(row["text"]))

    # ---- sample 2: decisions dated 2006-2025, minus every sample-1 case ----
    year = pd.to_datetime(df["document_date_en"], errors="coerce", utc=True).dt.year
    pool2 = df[(year >= 2006) & (year <= 2025) & (df.text.str.len() > 0)]
    if d == "FC":
        pool2 = pool2[pool2.text.apply(lambda t: bool(FC_FILTER.search(t)))]
    pool2 = pool2[~pool2.index.isin(s1.index)]          # exclusion by exact dataset row
    s2 = pool2.sample(n=SIZES[d], random_state=SEED)
    for j, (idx, row) in enumerate(s2.iterrows()):
        record(rows2, f"{START2[d] + j:03d}_{d}", d, idx, row, str(row["text"]))
    del df

manifest1, manifest2 = pd.DataFrame(rows1), pd.DataFrame(rows2)
# prod_uid links each sample-2 case to its file in the full-corpus run (Appendices 4-5)
manifest2["prod_uid"] = manifest2.dataset + "_" + manifest2.orig_idx.astype(str).str.zfill(5)
print(f"\nsample 1: {len(manifest1)} cases | sample 2: {len(manifest2)} cases")

RAD: 14,156 decisions in the corpus


RPD: 6,729 decisions in the corpus


RLLR: 927 decisions in the corpus


FC: 35,738 decisions in the corpus



sample 1: 300 cases | sample 2: 300 cases


In [3]:
print("draw 1 by source:", manifest1.dataset.value_counts().to_dict(), "| draw 2 by source:", manifest2.dataset.value_counts().to_dict())

draw 1 by source: {'FC': 100, 'RAD': 67, 'RPD': 67, 'RLLR': 66} | draw 2 by source: {'FC': 100, 'RAD': 67, 'RPD': 67, 'RLLR': 66}


### Verification against the committed reference file

The rebuilt samples must match the committed reference file,
`data/golden_ai_generated_sample.json` (600 rows: sample metadata plus reference labels), case
for case.

In [4]:
gold = pd.read_json(DATA / "golden_ai_generated_sample.json")   # the committed reference file

chk1 = gold[["case_id", "text_chars"]].merge(manifest1[["case_id", "text_chars"]],
                                             on="case_id", suffixes=("_repo", "_rebuilt"))
assert len(chk1) == 300 and (chk1.text_chars_repo == chk1.text_chars_rebuilt).all()

chk2 = gold[["case_id", "prod_uid"]].merge(manifest2[["case_id", "prod_uid"]],
                                           on="case_id", suffixes=("_repo", "_rebuilt"))
assert len(chk2) == 300 and (chk2.prod_uid_repo == chk2.prod_uid_rebuilt).all()

print("Both samples reproduced exactly: all 600 rebuilt cases match the committed reference file.")

Both samples reproduced exactly: all 600 rebuilt cases match the committed reference file.


### What the benchmark was drawn from, documented after the fact

The draws above are frozen. This cell records two facts about them that later appendices rely
on: that every benchmark case has decision text (draw 1 did not require it for the tribunal
sources), and how many benchmark cases fall outside the 2006–2025 study window — and therefore
outside the production corpus of Appendix 3, whose decisions are linked here by production
`uid` (draw 2) or `(source, citation)` (draw 1).

In [5]:
assert (gold.text_chars > 0).all()
print(f"decision text present for all {len(gold)} cases (shortest: {gold.text_chars.min():,} characters)")

year = pd.to_datetime(gold.document_date, utc=True).dt.year
outside = gold[(year < 2006) | (year > 2025)]
print(f"benchmark cases dated outside 2006-2025: {len(outside)} {outside.groupby('dataset').size().to_dict()}, "
      f"years {sorted(year[outside.index].unique().tolist())}; all from draw 1: "
      f"{bool(outside.case_id.str[:3].astype(int).lt(300).all())}")

prod = pd.read_json(DATA / "production_codes.json")   # the production corpus (Appendix 3)
prod_keys = set(zip(prod.dataset, prod.citation.astype(str)))
prod_uids = set(prod.uid)
in_corpus = [(u in prod_uids) if isinstance(u, str) else ((d, str(c)) in prod_keys)
             for u, d, c in zip(gold.prod_uid, gold.dataset, gold.citation)]
absent = gold[[not x for x in in_corpus]]
print(f"benchmark cases with no record in the production corpus: {len(absent)} "
      f"{absent.groupby('dataset').size().to_dict()}; the same cases as the out-of-window set: "
      f"{set(absent.case_id) == set(outside.case_id)}")

decision text present for all 600 cases (shortest: 2,037 characters)
benchmark cases dated outside 2006-2025: 15 {'FC': 10, 'RPD': 5}, years [2002, 2003, 2004, 2005, 2026]; all from draw 1: True
benchmark cases with no record in the production corpus: 15 {'FC': 10, 'RPD': 5}; the same cases as the out-of-window set: True


In [6]:
WRITE_CASE_FILES = False   # set True to write the 600 decision texts to data/cases/
                           # (needed only to re-run the model yourself in Appendix 2)
if WRITE_CASE_FILES:
    (DATA / "cases").mkdir(parents=True, exist_ok=True)
    for m in (manifest1, manifest2):
        for cid in m.case_id:
            (DATA / "cases" / f"{cid}.txt").write_text(texts[cid], encoding="utf-8")
    print("case text files written")

## How the reference labels were created

Each decision was coded by a Claude Fable 5 agent (running in Claude Code) that read it
in full — one case at a time, roughly ten cases per agent — applying the written rubric in
`data/golden_sample_coding_rubric.md`.

**The coded questions**:

| Question | Codes |
|---|---|
| `timing` — was the claim made more than 1 year after the principal claimant's first entry to Canada? | a = yes, clearly · b = no, clearly (≤ 1 year) · c = unclear |
| `outcome` — was the principal claimant successful? | a = yes · b = no · c = unclear |
| `basis` — what persecution was *alleged* (not what was proven)? | a = gender · b = SOGIE · c = both · d = other · e = unclear |
| `jr_refugee` (Federal Court cases only, asked first) — is this a judicial review of an RPD/RAD refugee determination? | a = yes · b = no · c = unknown — only “a” cases get the three main codes |

For each case the agent wrote one small JSON file with the codes plus a one-sentence evidence
field per code, quoting or paraphrasing the supporting passage — so any coding can be audited
against the decision text. Federal Court cases answered the `jr_refugee` gate first; cases that
are not refugee judicial reviews got `null` for the three main codes.

A validation script checked every label file (presence, JSON shape, allowed code values, the FC
gating rule) — all 600 passed — and compiled the labels with the sample metadata into the
reference file used throughout these appendices, `data/golden_ai_generated_sample.json`. One
example label file is committed as `data/example_golden_label.json` (shown below); the full set
of 600 is part of the study's research archive.

> **What the benchmark is — and is not.** The benchmark coding is much stronger than the
> production model being evaluated, but it is still *machine* coding by a frontier model, not
> coding by human experts. It measures whether the smaller model can stand in for a much stronger
> one, not accuracy against a human gold standard. Wherever these appendices say “accuracy,” read
> “agreement with the benchmark labels.” The human step in the study is the *reasonableness
> review* of production outputs (Appendix 4).


In [7]:
example = DATA / "example_golden_label.json"
print("example benchmark label:\n")
print(json.dumps(json.loads(example.read_text(encoding="utf-8")), indent=2))

example benchmark label:

{
  "case_id": "000_RAD",
  "timing": "b",
  "timing_evidence": "The Appellant arrived in Montreal in 2014 on a false passport and made her refugee claim at the port of entry after customs officials discovered her true identity, so the claim was made immediately upon her first entry.",
  "outcome": "b",
  "outcome_evidence": "The RAD confirmed the RPD's rejection of the claim: 'Pursuant to paragraph 111(1)(a) of the IRPA, I confirm the decision of the RPD. The appeal is denied.'",
  "basis": "a",
  "basis_evidence": "She alleged domestic violence by her ex-husband and abduction and sex trafficking/sexual slavery by her ex-boyfriend (gender-based persecution), along with discrimination based on her Albanian ethnicity; no SOGIE allegations."
}


## The reference labels at a glance

Distributions of the labels across the 600 cases. `(gated out)` codes are Federal Court cases
that failed the `jr_refugee` gate, so the three main questions do not apply.

In [8]:
gold = pd.read_json(DATA / "golden_ai_generated_sample.json")
print(f"{len(gold)} cases in the benchmark sample\n")
for q in ["timing", "outcome", "basis"]:
    print(f"--- {q} ---")
    print(pd.crosstab(gold.dataset, gold[q].fillna("(gated out)"), margins=True), "\n")

600 cases in the benchmark sample

--- timing ---
timing   (gated out)   a    b    c  All
dataset                                
FC                45   9   96   50  200
RAD                0  13   76   45  134
RLLR               0  12   65   55  132
RPD                0  12  100   22  134
All               45  46  337  172  600 

--- outcome ---
outcome  (gated out)    a    b  c  All
dataset                               
FC                45   60   94  1  200
RAD                0   41   92  1  134
RLLR               0  132    0  0  132
RPD                0    5  127  2  134
All               45  238  313  4  600 

--- basis ---
basis    (gated out)    a   b   c    d   e  All
dataset                                        
FC                45   31   8   2  105   9  200
RAD                0   26  11   3   86   8  134
RLLR               0   27  35   3   67   0  132
RPD                0   23  10   3   95   3  134
All               45  107  64  11  353  20  600 



Appendix 2 uses these labels as the reference standard for validating the production model.